In [8]:
# Songsterr API API Example
from requests.sessions import Session
import requests

In [16]:
s = requests.Session()
URL = "http://www.songsterr.com/a/wa/bestMatchForQueryString"
PARAMS = {'s' : "Nothing's Gonna Hurt You Baby", 'a' : "Cigarettes After Sex"}
R = s.get(url=URL, params=PARAMS)
# R.content

In [22]:
print(R.content)

b'<!doctype html>\n<html lang="en" prefix="og: http://ogp.me/ns# fb: http://ogp.me/ns/fb# music: http://ogp.me/ns/music#" color-scheme="auto">\n<head>\n<meta charSet="utf-8">\n<meta http-equiv="x-ua-compatible" content="ie=edge">\n\n<title>Nothing\'s Gonna Hurt You Baby Solo Tab by Cigarettes After Sex | Songsterr Tabs with Rhythm</title>\n<meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover">\n\n<link href="https://static2.songsterr.com/production-main/static2/latest/ShowroomHeaderLink-5587039b24.css" media="all" rel="stylesheet" type="text/css" crossorigin="anonymous"><link href="https://static2.songsterr.com/production-main/static2/latest/chordsWebviewClient-baf92eadd8.css" media="all" rel="stylesheet" type="text/css" crossorigin="anonymous"><link href="https://static2.songsterr.com/production-main/static2/latest/ShowroomFooter-6afa17c20c.css" media="all" rel="stylesheet" type="text/css" crossorigin="anonymous"><link href="https://static2.songsterr.

In [ ]:
R.

{'Server': 'nginx', 'Date': 'Wed, 17 Sep 2025 08:00:50 GMT', 'Content-Type': 'text/html; charset=utf-8', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'vary': 'Accept-Language, accept-encoding', 'link': '<https://static2.songsterr.com/production-main/static2/latest/ShowroomHeaderLink-5587039b24.css>; rel=preload; crossorigin=anonymous; as=style,<https://static2.songsterr.com/production-main/static2/latest/chordsWebviewClient-baf92eadd8.css>; rel=preload; crossorigin=anonymous; as=style,<https://static2.songsterr.com/production-main/static2/latest/ShowroomFooter-6afa17c20c.css>; rel=preload; crossorigin=anonymous; as=style,<https://static2.songsterr.com/production-main/static2/latest/appClient-e2cea1d50d.css>; rel=preload; crossorigin=anonymous; as=style,<https://static2.songsterr.com/production-main/static2/latest/ShowroomPlaceholder.module-658a84a68e.css>; rel=preload; crossorigin=anonymous; as=style,<https://static2.songsterr.com/production-main/static2/latest/SentryLaz

In [18]:
type(R.content)

bytes

### CHatGPT 

In [20]:
def parse_bass_tab(html_text):
    """
    Given the HTML of a tab page, parse and extract the bass track/tab lines.
    Returns a dict of e.g. string -> list of fret numbers, or text lines.
    """
    soup = BeautifulSoup(html_text, "html.parser")

    # Find the track(s) section. Songsterr HTML is complicated,
    # but generally there are divs or sections for each track (bass, guitar, etc.)
    # We need to find the bass track.

    # Example approach: find a track element whose name / label contains "Bass"
    track_divs = soup.find_all("div", class_="track")  # adjust class depending on HTML
    bass_div = None
    for td in track_divs:
        name_tag = td.find("span", class_="track-name")  # or similar label
        if name_tag and "bass" in name_tag.text.lower():
            bass_div = td
            break

    if bass_div is None:
        print("No bass track found in this tab.")
        return None

    # Once we have the bass track, get the tab lines/text
    # Often tabs are rendered as preformatted text, or with <pre> or <code> tags
    tab_pre = bass_div.find("pre") or bass_div.find("code")
    if not tab_pre:
        # fallback: maybe lines with class etc.
        # Try to find all lines under that div
        lines = [line.get_text() for line in bass_div.find_all("div", class_="tab-line")]
    else:
        raw = tab_pre.get_text()
        lines = raw.splitlines()

    # Parse each line: often there are strings represented with lines like
    # G|-------| etc. or sometimes tab with rhythm info.
    # For now, just return the lines.
    return {
        "lines": lines
    }



In [21]:
parse_bass_tab(R.content)

No bass track found in this tab.


In [13]:
import requests
from bs4 import BeautifulSoup
import re

def search_song(song_name):
    """
    Searches Songsterr API for a song name,
    returns a list of candidate songs (metadata).
    """
    url = "https://www.songsterr.com/a/ra/songs.json"
    params = {"pattern": song_name}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.json()  # list of songs

def best_match(song_name):
    """
    Get the best matching song (metadata) for the name.
    """
    url = "https://www.songsterr.com/a/wa/bestMatchForQueryStringPart"
    params = {"s": song_name}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.json()  # often a list; pick first

def fetch_tab_html(song_id):
    """
    Fetches the HTML of the tab view for a given Songsterr song ID.
    """
    url = "https://www.songsterr.com/a/wa/view"
    params = {"r": song_id}
    resp = requests.get(url, params=params)
    resp.raise_for_status()
    return resp.text  # HTML

def parse_bass_tab(html_text):
    """
    Given the HTML of a tab page, parse and extract the bass track/tab lines.
    Returns a dict of e.g. string -> list of fret numbers, or text lines.
    """
    soup = BeautifulSoup(html_text, "html.parser")

    # Find the track(s) section. Songsterr HTML is complicated,
    # but generally there are divs or sections for each track (bass, guitar, etc.)
    # We need to find the bass track.

    # Example approach: find a track element whose name / label contains "Bass"
    track_divs = soup.find_all("div", class_="track")  # adjust class depending on HTML
    bass_div = None
    for td in track_divs:
        name_tag = td.find("span", class_="track-name")  # or similar label
        if name_tag and "bass" in name_tag.text.lower():
            bass_div = td
            break

    if bass_div is None:
        print("No bass track found in this tab.")
        return None

    # Once we have the bass track, get the tab lines/text
    # Often tabs are rendered as preformatted text, or with <pre> or <code> tags
    tab_pre = bass_div.find("pre") or bass_div.find("code")
    if not tab_pre:
        # fallback: maybe lines with class etc.
        # Try to find all lines under that div
        lines = [line.get_text() for line in bass_div.find_all("div", class_="tab-line")]
    else:
        raw = tab_pre.get_text()
        lines = raw.splitlines()

    # Parse each line: often there are strings represented with lines like
    # G|-------| etc. or sometimes tab with rhythm info.
    # For now, just return the lines.
    return {
        "lines": lines
    }

def get_bass_tab(song_name):
    # Search & pick best match
    match = best_match(song_name)
    if not match:
        raise ValueError(f"No song match found for '{song_name}'")

    # match could be a list or single dict
    if isinstance(match, list):
        song = match[0]
    else:
        song = match

    song_id = song.get("id")
    if song_id is None:
        raise ValueError("Song metadata did not return an ID")

    html = fetch_tab_html(song_id)
    parsed = parse_bass_tab(html)
    return {
        "song": song,
        "bass_tab": parsed
    }

if __name__ == "__main__":
    song_name = input("Enter song name: ")
    result = get_bass_tab(song_name)
    print("Song metadata:", result["song"])
    print("Bass tab lines:")
    for l in result["bass_tab"]["lines"]:
        print(l)


HTTPError: 404 Client Error: Not Found for url: https://www.songsterr.com/a/wa/bestMatchForQueryStringPart?s=Nothing%27s+Gonna+Hurt+You+Baby